# Micropolar Piezoelectric Actuator — Topology Optimisation (MMA/NLopt) using Gridap

In [1]:
# FEA
using Gridap
using Gridap.Geometry
using Gridap.Fields
using Gridap.TensorValues
using Gridap.CellData
using Gridap.FESpaces
using GridapTopOpt

# Meshing
using Gmsh
using GridapGmsh

using LinearAlgebra
using SparseArrays

# Gradient (adjoint)
using ChainRulesCore, Zygote
import ChainRulesCore: rrule

# Optimiser — MMA via NLopt
using NLopt
using Printf

In [2]:
# ── Geometry ─────────────────────────────────────────────────────────────────
const L  = 1e-2*1e3          # plate length (m)
const hf = L / 150       # element size

# ── Physical constants ────────────────────────────────────────────────────────
h  = 1e-4*1e3                # plate thickness (m)
ϕ  = 500.0                 # applied voltage (V)
ks = 5000e3*1e-3               # spring stiffness at output DOF (N/m)

Ez = ϕ / h               # through-thickness electric field (V/m)

penal_uu = 3             # stiffness penalisation
penal_uφ = 4             # coupling penalisation  
penal_P  = 1             # polarisation penalisation

# ── Optimisation settings ─────────────────────────────────────────────────────
volfrac  = (1 - (1/2)*(3/4))*(0.5) # Since 40 % of actua domain after removing passive region

0.3125

In [3]:
const E_mod = 7.629e10*1e-6    # Young's modulus (Pa)
const ν_mod = 0.2478      # Poisson's ratio
const G     = E_mod / (2*(1 + ν_mod))

const l = L/60         
const N = 0.0        

const λₘₐₜ = E_mod * ν_mod / (1 - ν_mod^2)        # plane stress
const κₘₐₜ = 2*G * N^2 / (1 - N^2)
const μₘₐₜ = G * (1 - 2*N^2) / (1 - N^2)
const γₘₐₜ = 4*G * l^2

const E_Matrx = TensorValue(0, 1, -1, 0)

function σ_Bmod(ε)
    σM = ((λₘₐₜ)*tr(ε)*one(ε) + (2*μₘₐₜ + κₘₐₜ)*(ε))
    return σM
end

E_Matrx = TensorValue(0,1,-1,0)

function ε_Skw(∇,θ)
    ∇ᵀ = transpose(∇)
    w = (0.5*(∇ᵀ - ∇)) - (E_Matrx*θ)
    return w
end

function σ_Cmod(ϵ_skew)
    σM = κₘₐₜ*ϵ_skew
    return σM
end

function M_mod(∇)
    M = γₘₐₜ*∇
    return M
end

function Skw(u,θ)
    ∇ᵀ = transpose(∇(u))
    w = (0.5*(∇ᵀ - ∇(u)) - (E_Matrx*θ))
    return w
end

Skw (generic function with 1 method)

In [4]:
nx = 150
ny = 150
domain = (0,L,0,L)
model = CartesianDiscreteModel(domain,(nx,ny))
f_Γ_D1(x) = (x[1] ≈ 0.0) && (0 <= x[2] <= 0.25*L)
f_Γ_D2(x) = (x[2] ≈ L) && (0 <= x[1] <= 2*L/nx)

update_labels!(1,model,f_Γ_D1,"LeftFixedBoundary")
update_labels!(2,model,f_Γ_D2,"SpringBoundary")

In [5]:
elemsize = maximum(get_el_Δ(model))
r  = 2 * elemsize #(9 * elemsize) / (2 * sqrt(3))   # filter radius
β  = 4                                 # threshold sharpness
η  = 0.5                               # threshold centre

0.5

In [6]:
path = "./MicropolarActuatorPlate_Pusher_In_mm/r_$(r)/ks$(ks)/N_$(N)/l_$(l)/"
mkpath(path)
cd(path)

In [7]:
writevtk(model, "MicropolarActuatorPlate")

3-element Vector{Vector{String}}:
 ["MicropolarActuatorPlate_0.vtu"]
 ["MicropolarActuatorPlate_1.vtu"]
 ["MicropolarActuatorPlate_2.vtu"]

### Piezoelectric constitutive tensors (PZT-4, plane stress) — unchanged from elastic version

In [8]:
C11, C12, C33 = 9.1187e10, 3.0025e10, 3.0581e10
e31 = -14.9091*1e-3

function voigt2tensor4(A::Array{M,2}) where M
    if isequal(size(A), (3,3))
        return SymFourthOrderTensorValue(
            A[1,1], A[3,1], A[2,1],
            A[1,3], A[3,3], A[2,3],
            A[1,2], A[3,2], A[2,2])
    elseif isequal(size(A), (6,6))
        return SymFourthOrderTensorValue(
            A[1,1], A[6,1], A[5,1], A[2,1], A[4,1], A[3,1],
            A[1,6], A[6,6], A[5,6], A[2,6], A[4,6], A[3,6],
            A[1,5], A[6,5], A[5,5], A[2,5], A[4,5], A[3,5],
            A[1,2], A[6,2], A[5,2], A[2,2], A[4,2], A[3,2],
            A[1,4], A[6,4], A[5,4], A[2,4], A[4,4], A[3,4],
            A[1,3], A[6,3], A[5,3], A[2,3], A[4,3], A[3,3])
    end
end

function voigt2tensor3(A::Array{M,2}) where M
    if isequal(size(A), (2,3))
        # e[i,j,k]: Voigt: 1→(1,1), 2→(2,2), 3→(1,2)=(2,1) symmetric
        return ThirdOrderTensorValue(
            A[1,1], A[2,1],   # e[1,1,1], e[2,1,1]
            A[1,3], A[2,3],   # e[1,2,1], e[2,2,1]
            A[1,3], A[2,3],   # e[1,1,2], e[2,1,2]  (j↔k symmetry)
            A[1,2], A[2,2])   # e[1,2,2], e[2,2,2]
    end
end

function PZT4_2D()
    C_Voigt = [C11  C12  0.0;
               C12  C11  0.0;
               0.0  0.0  C33]
    e_Voigt = [e31  e31  0.0;
               0.0  0.0  0.0]
    C_piezo = voigt2tensor4(C_Voigt)
    e_piezo = voigt2tensor3(e_Voigt)
    return C_piezo, e_piezo
end

C_piezo, e_piezo = PZT4_2D()
Eⁱ = VectorValue(Ez, 0.0)

# Mechanical stress from piezo tensor (for reference, used in normalisation)
σ_mech_piezo(u) = C_piezo ⊙ ε(u)

# Piezoelectric stress — constant body load (uniform voltage through thickness)
σ_piezo = Eⁱ ⋅ e_piezo

TensorValue{2, 2, Float64, 4}(-74.5455, 0.0, 0.0, -74.5455)

### Micropolar constitutive parameters and operators


### SIMP interpolation — PEMAP-P scheme (unchanged)

In [9]:
const Emin = 1e-9
const eMin = 1e-9
const E0   = 1.0
const e0   = 1.0

function Em_uu(x)
    return Emin + (x ^ penal_uu) * (E0 - Emin)
end

function Em_uφ(x, P)
    return (eMin + (x ^ penal_uφ) * (e0 - eMin)) * (2*P - 1)^penal_P
end

Em_uφ (generic function with 1 method)

### Finite element spaces


In [10]:
order  = 1
degree = 2 * order

Ω  = Triangulation(model)
dΩ = Measure(Ω, degree)

# ── Displacement space (vector CG1) ──────────────────────────────────────────
reffe_Disp = ReferenceFE(lagrangian, VectorValue{2,Float64}, order)
V0_Disp = TestFESpace(model, reffe_Disp;
    conformity      = :H1,
    dirichlet_tags  = ["LeftFixedBoundary"],
    dirichlet_masks = [(true, true)] )

U0_Disp = TrialFESpace(V0_Disp, [VectorValue(0.0, 0.0)])

# ── Microrotation space (scalar CG1) — NEW for micropolar ────────────────────
reffe_theta = ReferenceFE(lagrangian, Float64, order)
V0_theta = TestFESpace(model, reffe_theta;
    conformity      = :H1,
    dirichlet_tags  = ["LeftFixedBoundary"],
    dirichlet_masks = [true])
U0_theta = TrialFESpace(V0_theta, [0.0])

# ── Multifield spaces [u; θ] ──────────────────────────────────────────────────
V0 = MultiFieldFESpace([V0_Disp, V0_theta])
U0 = MultiFieldFESpace([U0_Disp, U0_theta])

labels = get_face_labeling(model)
LoadTagId = get_tag_from_name(labels ,"SpringBoundary")
Γ_Load = BoundaryTriangulation(model ;tags = LoadTagId)
dΓ_Load = Measure(Γ_Load ,degree)
n_Γ_Load = get_normal_vector(Γ_Load) 
vol_Γ_Load   = sum(∫(1) * dΓ_Load)   # length of spring patch

# ── Design variable spaces (DG0 — one DOF per element) ───────────────────────
p_reffe = ReferenceFE(lagrangian, Float64, 0)
Q  = TestFESpace(Ω, p_reffe; vector_type = Vector{Float64})
P  = Q
np = num_free_dofs(P)

# ── Filtered density space (CG1 — continuous, for Helmholtz filter) ───────────
pf_reffe = ReferenceFE(lagrangian, Float64, 1)
Qf = TestFESpace(Ω, pf_reffe; vector_type = Vector{Float64})
Pf = Qf

# ── Bundle (passive_void added after it is built below) ──────────────────────
fem_params = (; V0_Disp, U0_Disp, V0_theta, U0_theta, V0, U0, Q, P, Qf, Pf, np, Ω, dΩ, n_Γ_Load, dΓ_Load,vol_Γ_Load)

println("Number of design elements: np = $np")
println("Displacement DOFs : $(num_free_dofs(U0_Disp))")
println("Rotation DOFs     : $(num_free_dofs(U0_theta))")
println("Total DOFs        : $(num_free_dofs(U0))")

# ── Passive void region: top-right corner (L-L/10, L/2-L/20) → (L, L/2) ─────
# CartesianDiscreteModel cells: x (column) fast, y (row) slow
#   linear_cell_id = (iy - 1)*nx + ix
# x-range: ix = 181 … 200  →  x ∈ [9L/10, L]
# y-range: iy = 96  … 100  →  y ∈ [9L/20, L/2]
function passive_void_from_region(model, nx, ny, x_min, x_max, y_min, y_max)
    dx = (model.grid.node_coords[end][1] - model.grid.node_coords[1][1]) / nx
    dy = (model.grid.node_coords[end][2] - model.grid.node_coords[1][2]) / ny
    
    ix_min = round(Int, x_min / dx) + 1
    ix_max = round(Int, x_max / dx)
    iy_min = round(Int, y_min / dy) + 1
    iy_max = round(Int, y_max / dy)
    
    return Int[(iy - 1) * nx + ix for iy in iy_min:iy_max for ix in ix_min:ix_max]
end

# fully automatic from coordinates
passive_void = passive_void_from_region(model, nx, ny,
    0, 0.75*L,          # x range
    L/4, 0.75*L       # y range
)
println("Passive void elements: $(length(passive_void)) (expected 100)")

# ── Rebuild fem_params with passive_void included ────────────────────────────
fem_params = (; V0_Disp, U0_Disp, V0_theta, U0_theta, V0, U0, Q, P, Qf, Pf, np, Ω, dΩ, n_Γ_Load, dΓ_Load,vol_Γ_Load,
               passive_void)

Number of design elements: np = 22500
Displacement DOFs : 45526
Rotation DOFs     : 22763
Total DOFs        : 68289
Passive void elements: 8288 (expected 100)


(V0_Disp = UnconstrainedFESpace(), U0_Disp = TrialFESpace(), V0_theta = UnconstrainedFESpace(), U0_theta = TrialFESpace(), V0 = MultiFieldFESpace(), U0 = MultiFieldFESpace(), Q = UnconstrainedFESpace(), P = UnconstrainedFESpace(), Qf = UnconstrainedFESpace(), Pf = UnconstrainedFESpace(), np = 22500, Ω = BodyFittedTriangulation(), dΩ = GenericMeasure(), n_Γ_Load = GenericCellField(), dΓ_Load = GenericMeasure(), vol_Γ_Load = 0.1333333333333333, passive_void = [5701, 5702, 5703, 5704, 5705, 5706, 5707, 5708, 5709, 5710  …  16753, 16754, 16755, 16756, 16757, 16758, 16759, 16760, 16761, 16762])

### Output DOF — mid-right node (x-displacement)

In [11]:
function get_dof_for_node_component(V0_Disp, node_id, component)
    dof_to_node = V0_Disp.metadata.node_and_comp_to_dof
    return dof_to_node[node_id][component]   # index node first, then component
end


function node_id_from_coord(model, x_target, y_target)
    coords = model.grid.node_coords
    _, idx = findmin(p -> (p[1] - x_target)^2 + (p[2] - y_target)^2, coords)
    return LinearIndices(coords)[idx]   # convert CartesianIndex → linear Int
end

# Usage — fully automatic from coordinates
node_id = node_id_from_coord(model, 0, L)

coords = model.grid.node_coords
println("Node $node_id coords: ", coords[node_id])   # sanity check

DMDOF = get_dof_for_node_component(fem_params.V0_Disp, node_id, 2)
println("DMDOF = ", DMDOF)
DMDOF = Int(DMDOF)

Node 22651 coords: (0.0, 10.0)
DMDOF = 45226


45226

### Normalisation constants `k0` and `α0`

`k0` — max element entry of the **full micropolar** stiffness matrix (all 4 terms).  
`α0` — max element entry of the piezoelectric load vector (displacement block only, unchanged).


In [12]:
function get_normalisation_constants(fem_params)
    (; V0, U0, V0_Disp, Ω, dΩ) = fem_params

    # ── k0: max entry of micropolar stiffness ─────────────────────────────────
    (w_b, v_b) = get_fe_basis(V0)
    (u_b, θ_b) = get_trial_fe_basis(U0)

    a_micro((u,θ),(w,v)) = ∫(
          h * (ε(w) ⊙ (σ_Bmod ∘ (ε(u))))
        + h * (Skw(w,v) ⊙ (σ_Cmod ∘ (ε_Skw ∘ (∇(u), θ))))
        + h * (∇(v) ⋅ (M_mod ∘ (∇(θ))))
        - h * (v * (E_Matrx ⊙ (σ_Cmod ∘ (ε_Skw ∘ (∇(u), θ)))))
    )dΩ

    K_full  = assemble_matrix(a_micro, U0, V0)
    k0_val  = maximum(abs, K_full)

    # ── α0: max entry of piezo load vector (displacement test functions only) ─
    v_disp = get_fe_basis(V0_Disp)
    b_piezo_form = ∫( h * (ε(v_disp) ⊙ σ_piezo) ) * dΩ
    cell_vec = b_piezo_form[Ω]
    α0_val   = maximum(maximum(abs.(cell_vec[e])) for e in 1:num_cells(Ω))

    return k0_val, α0_val
end

k0, α0 = get_normalisation_constants(fem_params)
println("k0 = $k0")
println("α0 = $α0")
k0, α0 = 1,1 

k0 = 14913.447028642244
α0 = 0.24848499999999993


(1, 1)

### FE solver — `stepDisp2`

In [13]:
e2 = VectorValue(0.0, 1.0) 

VectorValue{2, Float64}(0.0, 1.0)

In [14]:
function stepDisp2(fem_params, ph, polh, DMDOF)
    (; U0, V0, dΩ) = fem_params
    a((u,θ),(w,v)) = ∫(((p -> Em_uu(p)) ∘ ph) * (1/k0) * (
                      h * (ε(w) ⊙ (σ_Bmod ∘ (ε(u))))
                    + h * ((Skw(w,v)) ⊙ (σ_Cmod ∘ (ε_Skw ∘ (∇(u), θ))))
                    + h * ((∇(v)) ⋅ (M_mod ∘ (∇(θ))))
                    - h * (v * ((E_Matrx) ⊙ (σ_Cmod ∘ (ε_Skw ∘ (∇(u), θ)))))))dΩ + ∫(ks * (((u ⋅ e2) * (w ⋅ e2))) ) * dΓ_Load 
    
    b((w,v)) = ∫((((p,P) -> Em_uφ(p,P)) ∘ (ph, polh)) * (-h * (1/α0) * (ε(w) ⊙ σ_piezo)))dΩ

    op  = AffineFEOperator(a, b, U0, V0)
    uh = solve(op)
    K   = get_matrix(op)
    F   = get_vector(op)
    return uh, K,F
end

stepDisp2 (generic function with 1 method)

### Helmholtz PDE filter and Heaviside threshold — **unchanged from elastic version**

In [15]:
a_f(r, u, v) = r^2 * (∇(v) ⋅ ∇(u))

function Filter(p0; r, fem_params)
    ph = FEFunction(fem_params.P, p0)
    op = AffineFEOperator(fem_params.Pf, fem_params.Qf) do u, v
        ∫(a_f(r, u, v))fem_params.dΩ + ∫(v * u)fem_params.dΩ,
        ∫(v * ph)fem_params.dΩ
    end
    pfh = solve(op)
    return get_free_dof_values(pfh)
end

function Threshold(pfh; β, η)
    return (tanh(β*η) + tanh(β*(pfh - η))) / (tanh(β*η) + tanh(β*(1.0 - η)))
end

Dptdpf(pf; β, η) = β*(1.0 - tanh(β*(pf - η))^2) / (tanh(β*η) + tanh(β*(1.0 - η)))

Dptdpf (generic function with 1 method)

### Sensitivity derivatives

In [16]:
# ── Scalar SIMP derivative functions
DEdpf_uu(pf; β, η) = penal_uu * Threshold(pf; β, η)^(penal_uu-1) * Dptdpf(pf; β, η) * (E0 - Emin)
DEdpf_uφ(pf; β, η) = penal_uφ * Threshold(pf; β, η)^(penal_uφ-1) * Dptdpf(pf; β, η) * (e0 - eMin)
DEdpol(P)           = penal_P  * (2*P - 1)^(penal_P-1) * 2 * (e0 - eMin)

function DAdpf(u, θ, w, v, pfh; β, η)
    return ((p -> DEdpf_uu(p; β, η)) ∘ pfh) * ((1/k0) * (h * (ε(w) ⊙ (σ_Bmod ∘ (ε(u))))
                    + h * ((Skw(w,v)) ⊙ (σ_Cmod ∘ (ε_Skw ∘ (∇(u), θ))))
                    + h * ((∇(v)) ⋅ (M_mod ∘ (∇(θ))))
                    - h * (v * ((E_Matrx) ⊙ (σ_Cmod ∘ (ε_Skw ∘ (∇(u), θ)))))))
end

function Db_dpf(λ_u, pfh, polh; β, η)
    return ((p -> DEdpf_uφ(p; β, η)) ∘ pfh) *
           ((P -> (2*P - 1)^penal_P) ∘ polh) *
           (-h / α0) * (ε(λ_u) ⊙ σ_piezo)
end

function Db_dpol(λ_u, pth, polh)
    return ((P -> DEdpol(P)) ∘ polh) *
           ((p -> p^penal_uφ) ∘ pth) *
           (-h / α0) * (ε(λ_u) ⊙ σ_piezo)
end

Db_dpol (generic function with 1 method)

In [17]:
function Dgdp(dgdpf; r, fem_params)
    Af = assemble_matrix(fem_params.Pf, fem_params.Qf) do u, v
        ∫(a_f(r, u, v))fem_params.dΩ + ∫(v * u)fem_params.dΩ
    end
    wvec = Af' \ dgdpf
    wh   = FEFunction(fem_params.Pf, wvec)
    l_temp(dp) = ∫(wh * dp)fem_params.dΩ
    return assemble_vector(l_temp, fem_params.P)
end

Dgdp (generic function with 1 method)

### Sanity check — full solid, uniform polarisation

In [18]:
p0_init   = ones(fem_params.np)
p0_init[fem_params.passive_void] .= 0.001   # passive void: start at ρ_min
pol0_init = fill(1.0, fem_params.np)

pf_vec_chk = Filter(p0_init; r, fem_params)
pfh_chk    = FEFunction(fem_params.Pf, pf_vec_chk)
pth_chk    = (pf -> Threshold(pf; β, η)) ∘ pfh_chk

polh_chk     = FEFunction(fem_params.P, pol0_init)
polh_eff_chk = Operation(P -> (2*P - 1)^penal_P)(polh_chk)

uh_comb_chk, K_mat,F = stepDisp2(fem_params, pth_chk, polh_chk, DMDOF)
uh_u_chk, uh_θ_chk = uh_comb_chk

filename1 = "SanityCheck_FullSolid_K$(replace(string(ks), "." => "_"))"
writevtk(Ω, filename1, cellfields = [
    "uh"      => uh_u_chk,
    "theta"   => uh_θ_chk,
    "pth"     => pth_chk,
    "pol"     => polh_chk,
    "pol_eff" => polh_eff_chk
])
println("Sanity-check VTK written: $(filename1).vtu")

uh_vec_chk = get_free_dof_values(uh_comb_chk)
println("Full-solid U_out = ", uh_vec_chk[DMDOF], " m")

Sanity-check VTK written: SanityCheck_FullSolid_K5000_0.vtu
Full-solid U_out = 1.5128516620174583e-5 m


In [19]:
u2_load = uh_u_chk ⋅ VectorValue(0.0, 1.0) 

avg_u2_sanity = sum(∫(u2_load)*dΓ_Load) / sum(∫(1.0)*dΓ_Load)

println("Average u2 on dΓ_Load = ", avg_u2_sanity, " m")

Average u2 on dΓ_Load = 8.053029576794702e-5 m


### Objective + gradient — `gf_p`


In [20]:
function gf_p(vars::Vector, grad::Vector; r, β, η, fem_params)
    np  = fem_params.np
    p0  = vars[1:np]
    pol = vars[np+1:end]

    pf_vec = Filter(p0; r, fem_params)
    pfh    = FEFunction(fem_params.Pf, pf_vec)
    pth    = (pf -> Threshold(pf; β, η)) ∘ pfh

    polh = FEFunction(fem_params.P, pol)

    uh_comb, K_mat, _ = stepDisp2(fem_params, pth, polh, DMDOF)
    uh_u, uh_θ = uh_comb

    U_out  = sum(∫((uh_u ⋅ e2)/vol_Γ_Load) * fem_params.dΓ_Load)
    gf_val = -U_out * 1e6

    if length(grad) > 0

        dfduFn(w) = ∫((-1e6 * (w ⋅ e2))/vol_Γ_Load) * fem_params.dΓ_Load
        dfduVec   = assemble_vector(dfduFn, fem_params.U0_Disp)

        dfdθVec = zeros(num_free_dofs(fem_params.U0_theta))

        dfduθVec = vcat(-dfduVec, -dfdθVec)

        wvec = K_mat' \ dfduθVec

        n_u   = num_free_dofs(fem_params.U0_Disp)
        wUvec = wvec[1:n_u]
        wθvec = wvec[n_u+1:end]

        λh_u = FEFunction(fem_params.U0_Disp, wUvec)
        λh_θ = FEFunction(fem_params.U0_theta, wθvec)

        # ── Density sensitivity ───────────────────────────────────────────────
        wtimsdgdp_density(dp) =
            ∫( DAdpf(uh_u, uh_θ, λh_u, λh_θ, pfh; β, η) * dp ) * fem_params.dΩ -
            ∫( Db_dpf(λh_u, pfh, polh; β, η) * dp ) * fem_params.dΩ

        dgfdpf       = assemble_vector(wtimsdgdp_density, fem_params.Pf)
        grad_density = Dgdp(dgfdpf; r, fem_params)

        # ── Polarisation sensitivity ──────────────────────────────────────────
        wtimsdgdp_pol(dpol) =
            ∫( (-1.0) * Db_dpol(λh_u, pth, polh) * dpol ) * fem_params.dΩ

        grad_pol = assemble_vector(wtimsdgdp_pol, fem_params.P)

        # ── Zero sensitivities for passive void elements ─────────────────────
        grad_density[fem_params.passive_void] .= 0.0
        grad_pol[fem_params.passive_void]     .= 0.0

        grad[:] = vcat(collect(grad_density), collect(grad_pol))
    end

    # ──VTK output ─────────────────────────────────────────────────────────
    polh_eff = Operation(P -> (2*P - 1)^penal_P)(polh)
    fname    = "OptResults_K$(replace(string(ks), "." => "_"))"
    writevtk(fem_params.Ω, fname, cellfields = [
        "uh"       => uh_u,
        "theta"    => uh_θ,
        "gradθ"    => ∇(uh_θ),
        "pth"      => pth,
        "polh"     => polh,
        "polh_eff" => polh_eff
    ])

    if length(grad) > 0
        grad_density_h = FEFunction(fem_params.P, collect(grad_density))
        grad_pol_h     = FEFunction(fem_params.P, collect(grad_pol))
        writevtk(fem_params.Ω, "grad_debug", cellfields = [
            "grad_density" => grad_density_h,
            "grad_pol"     => grad_pol_h
        ])
    end

    open("gvaluemma.txt", "a") do io
        write(io, "$gf_val\n")
    end

    return gf_val
end

gf_p (generic function with 1 method)

### Volume constraint — unchanged (depends only on density, not microrotation)

In [21]:
dv = get_array(∫(1)fem_params.dΩ)

function cf_p(vars::Vector, gradc::Vector; r, β, η, fem_params)
    np  = fem_params.np
    p0  = vars[1:np]

    pf_vec = Filter(p0; r, fem_params)
    pfh    = FEFunction(fem_params.Pf, pf_vec)

    if length(gradc) > 0
        gradc[:] = vcat(dv, zeros(np))
    end

    return sum(∫(pfh) * fem_params.dΩ) - volfrac * sum(dv)
end

cf_p (generic function with 1 method)

### Finite-difference gradient check

In [22]:
np_chk = fem_params.np

p0_fd   = fill(volfrac, np_chk)
pol0_fd = fill(1.0, np_chk)
vars0   = vcat(p0_fd, pol0_fd)

δ = 1e-3

# Density perturbation at element 30
δp_disp        = zeros(2 * np_chk)
δp_disp[150]    = δ

# Polarisation perturbation at element 30
δp_pol         = zeros(2 * np_chk)
δp_pol[np_chk + 30] = δ

grad  = zeros(2 * np_chk)
gradm = zeros(2 * np_chk)

g0      = gf_p(vars0,            grad;  r, β, η, fem_params)
g1_disp = gf_p(vars0 + δp_disp, gradm; r, β, η, fem_params)
g1_pol  = gf_p(vars0 + δp_pol,  gradm; r, β, η, fem_params)

println("==========================================")
println("FD CHECK — DENSITY")
println("==========================================")
println("FD approximation  : $(g1_disp - g0)")
println("Adjoint prediction: $(grad' * δp_disp)")
println("Relative error    : $((g1_disp - g0 - grad'*δp_disp) / (g1_disp - g0))")
println("==========================================")
println("FD CHECK — POLARISATION")
println("==========================================")
println("FD approximation  : $(g1_pol - g0)")
println("Adjoint prediction: $(grad' * δp_pol)")
println("Relative error    : $((g1_pol - g0 - grad'*δp_pol) / (g1_pol - g0))")
println("==========================================")

FD CHECK — DENSITY
FD approximation  : -1.4046586116478466e-9
Adjoint prediction: -1.4041088996145457e-9
Relative error    : 0.0003913492066630787
FD CHECK — POLARISATION
FD approximation  : 2.296493851972059e-6
Adjoint prediction: 2.296491151446671e-6
Relative error    : 1.1759340812473706e-6


### MMA optimiser — `gf_p_optimize`

In [23]:
function gf_p_optimize(vars_init::Vector; r, β, η, TOL=1e-4, MAX_ITER, fem_params)
    np = fem_params.np
    opt = Opt(:LD_MMA, 2 * np)

    passive_void = fem_params.passive_void
    lb_p = fill(0.001, np)
    ub_p = fill(1.0,   np)
    lb_p[passive_void] .= 0.001
    ub_p[passive_void] .= 0.001
    opt.lower_bounds = vcat(lb_p,       fill(0.0, np))
    opt.upper_bounds = vcat(ub_p,       fill(1.0, np))
    opt.maxeval      = MAX_ITER

    # ── iteration counter living in the closure ──────────────────────────────
    iter = Ref(0)

    opt.min_objective = (vars, grad) -> begin
        iter[] += 1
        # Tighten tolerance only after 100 iterations
        opt.xtol_rel = iter[] <= 100 ? 0.0 : TOL
        return gf_p(vars, grad; r, β, η, fem_params)
    end
    # ─────────────────────────────────────────────────────────────────────────

    inequality_constraint!(opt,
        (vars, gradc) -> cf_p(vars, gradc; r, β, η, fem_params), 1e-8)

    (g_opt, vars_opt, ret) = optimize(opt, vars_init)
    p_opt   = vars_opt[1:np]
    pol_opt = vars_opt[np+1:end]
    @show numevals = opt.numevals
    println("got $(g_opt) after $numevals iterations (returned $ret)")
    return g_opt, vars_opt, ret
end

gf_p_optimize (generic function with 1 method)

### Run the optimisation

In [24]:
np_run = fem_params.np

p0_run   = fill(volfrac, np_run)
p0_run[fem_params.passive_void] .= 0.001   # passive void: start at ρ_min
pol0_run = fill(1.0, np_run)

vars0_run = vcat(p0_run, pol0_run)

TOL      = 1e-10
MAX_ITER = 1500

g_opt, vars_opt, ret = gf_p_optimize(vars0_run; r, β, η, TOL, MAX_ITER, fem_params)

p_opt   = vars_opt[1:np_run]
pol_opt = vars_opt[np_run+1:end]

@printf "Final density  ∈ [%.4f, %.4f]\n" minimum(p_opt)   maximum(p_opt)
@printf "Final pol      ∈ [%.4f, %.4f]\n" minimum(pol_opt) maximum(pol_opt)

numevals = opt.numevals = 1500
got -1062.0260241426752 after 1500 iterations (returned MAXEVAL_REACHED)
Final density  ∈ [0.0010, 1.0000]
Final pol      ∈ [0.0000, 1.0000]


### Post-processing — final VTK with sharp threshold

In [25]:
β_post = 64    # sharper threshold for crisp boundary visualisation

pf_vec_opt = Filter(p_opt; r, fem_params)
pfh_opt    = FEFunction(fem_params.Pf, pf_vec_opt)
pth_opt    = (pf -> Threshold(pf; β=β_post, η)) ∘ pfh_opt

polh_opt     = FEFunction(fem_params.P, pol_opt)
polh_eff_opt = Operation(P -> (2*P - 1)^penal_P)(polh_opt)

uh_comb_opt, _ = stepDisp2(fem_params, pth_opt, polh_opt, DMDOF)
uh_u_opt, uh_θ_opt = uh_comb_opt

fname_final = "FinalResult_K$(replace(string(ks), "." => "_"))"
writevtk(Ω, fname_final, cellfields = [
    "uh"       => uh_u_opt,
    "theta"    => uh_θ_opt,
    "pth"      => pth_opt,
    "pol_opt"  => polh_opt,
    "pol_eff"  => polh_eff_opt
])
println("Final VTK written: $(fname_final).vtu")
println("Final U_out = ", get_free_dof_values(uh_comb_opt)[DMDOF], " m")

Final VTK written: FinalResult_K5000_0.vtu
Final U_out = 0.001488940061083151 m


In [26]:
u2_load_opt = uh_u_opt ⋅ VectorValue(0.0, 1.0) 

avg_u2_opt = sum(∫(u2_load_opt)*dΓ_Load) / sum(∫(1.0)*dΓ_Load)

println("Average u2 on dΓ_Load after optimization = ", avg_u2_opt, " m")

Average u2 on dΓ_Load after optimization = 0.001385206412336397 m


In [27]:
# ── Post-processing ──
N_post = 0.5 
l_post = L/10 

# ── Recompute for N_post, l_post ──
κₘₐₜ_post = 2*G * N_post^2 / (1 - N_post^2)
μₘₐₜ_post = G * (1 - 2*N_post^2) / (1 - N_post^2)
γₘₐₜ_post = 4*G * l_post^2

# ── New constitutive functions using the post N, l ──
function σ_Bmod_post(ε)
    return (λₘₐₜ)*tr(ε)*one(ε) + (2*μₘₐₜ_post + κₘₐₜ_post)*(ε)
end

function σ_Cmod_post(ϵ_skew)
    return κₘₐₜ_post*ϵ_skew
end

function M_mod_post(∇)
    return γₘₐₜ_post*∇
end

function stepDisp2_post(fem_params, ph, polh, DMDOF)
    (; U0, V0, dΩ) = fem_params
    a((u,θ),(w,v)) = ∫(((p -> Em_uu(p)) ∘ ph) * (1/k0) * (
                      h * (ε(w) ⊙ (σ_Bmod_post ∘ (ε(u))))
                    + h * ((Skw(w,v)) ⊙ (σ_Cmod_post ∘ (ε_Skw ∘ (∇(u), θ))))
                    + h * ((∇(v)) ⋅ (M_mod_post ∘ (∇(θ))))
                    - h * (v * ((E_Matrx) ⊙ (σ_Cmod_post ∘ (ε_Skw ∘ (∇(u), θ)))))))dΩ

    b((w,v)) = ∫((((p,P) -> Em_uφ(p,P)) ∘ (ph, polh)) * (-h * (1/α0) * (ε(w) ⊙ σ_piezo)))dΩ

    op  = AffineFEOperator(a, b, U0, V0)
    K   = get_matrix(op)
    F   = get_vector(op)

    K[DMDOF, DMDOF] += ks

    uh_vec = K \ F
    uh     = FEFunction(U0, uh_vec)
    return uh, K, F
end

β_post_field = 64
pf_vec_opt_post = Filter(p_opt; r, fem_params)
pfh_opt_post    = FEFunction(fem_params.Pf, pf_vec_opt_post)
pth_opt_post    = (pf -> Threshold(pf; β=β_post_field, η)) ∘ pfh_opt_post

polh_opt_post     = FEFunction(fem_params.P, pol_opt)
polh_eff_opt_post = Operation(P -> (2*P - 1)^penal_P)(polh_opt_post)

# ── Solve at the new N, l — same optimized topology ──
uh_comb_opt_post, _ = stepDisp2_post(fem_params, pth_opt_post, polh_opt_post, DMDOF)
uh_u_opt_post, uh_θ_opt_post = uh_comb_opt_post

fname_final_post = "FinalResult_K$(replace(string(ks), "." => "_"))" *
                    "_N$(replace(string(N_post), "." => "_"))" *
                    "_l$(replace(string(l_post), "." => "_"))"

writevtk(Ω, fname_final_post, cellfields = [
    "uh"       => uh_u_opt_post,
    "theta"    => uh_θ_opt_post,
    "pth"      => pth_opt_post,
    "pol_opt"  => polh_opt_post,
    "pol_eff"  => polh_eff_opt_post
])

println("Post-processing at N = $(N_post), l = $(l_post)  (original design; original N = $(N), l = $(l))")
println("Final VTK written: $(fname_final_post).vtu")
println("Final U_out = ", get_free_dof_values(uh_comb_opt_post)[DMDOF], " m")

Post-processing at N = 0.5, l = 1.0  (original design; original N = 0.0, l = 0.16666666666666666)
Final VTK written: FinalResult_K5000_0_N0_5_l1_0.vtu
Final U_out = 0.0001814143236715789 m


In [28]:
u2_load_opt_post = uh_u_opt_post ⋅ VectorValue(0.0, 1.0) 

avg_u2_opt_post = sum(∫(u2_load_opt_post)*dΓ_Load) / sum(∫(1.0)*dΓ_Load)

println("Average u2 on dΓ_Load after optimization = ", avg_u2_opt_post, " m")

Average u2 on dΓ_Load after optimization = 0.00037097778902608187 m


In [29]:
open("u2_load_summary.txt", "a") do io
    write(io, "ks = $ks:\n")
    write(io, "  avg_u2 (sanity)     = $avg_u2_sanity m\n")
    write(io, "  avg_u2 (opt)        = $avg_u2_opt m\n")
    write(io, "  avg_u2 (opt, post)  = $avg_u2_opt_post m\n\n")
end

50